# From Individuals to Populations: Bridging Mendelian and Population Genetics

## Making the Statistical Leap

**Authors:** Susama Kar & Dr. Alok Patel  
**Institution:** Department of Zoology, Kuchinda College, Sambalpur University

---

## 🤔 The Conceptual Challenge

**Students often ask:**
- "Why did we suddenly switch from genotypes to frequencies?"
- "How does Hardy-Weinberg relate to what Mendel discovered?"
- "What does 'random mating' actually look like?"
- "Why do some populations change and others don't?"

**By the end of this notebook, you'll understand:**
1. 🧬 How Mendelian ratios become population frequencies
2. 📊 What Hardy-Weinberg equilibrium really means
3. 🎲 Why populations are statistical, not deterministic
4. 🔄 How evolutionary forces change populations
5. 🎚️ Everything through interactive exploration!

---

## 🎯 Two Ways of Thinking

### Mendelian Genetics (Individual Focus)
**Questions:**
- What is THIS organism's genotype?
- What will THIS cross produce?
- What is the ratio of offspring?

**Example:**
```
Parents: Aa × Aa
Offspring: 1 AA : 2 Aa : 1 aa
```

### Population Genetics (Group Focus)
**Questions:**
- What proportion of alleles are A vs a?
- How do frequencies change over time?
- Is this population evolving?

**Example:**
```
Population: p(A) = 0.6, q(a) = 0.4
Genotypes: AA=0.36, Aa=0.48, aa=0.16
```

### The Bridge
**Population genetics is just Mendelian genetics applied to MANY individuals and asking statistical questions!**


In [ ]:
# Setup and Imports

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Rectangle, Circle
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider, IntSlider, Dropdown, Checkbox, RadioButtons
import pandas as pd
import seaborn as sns
from scipy.stats import binom, chi2
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')

# Set random seed for reproducibility
np.random.seed(42)

print("✓ All libraries imported successfully!")
print("\n🎯 Ready to bridge from Mendelian to Population Genetics!")

---

## 🧬 Part 1: From One Cross to Many - The Emergence of Frequencies

### The Transformation

**Start:** One Mendelian cross  
**Scale up:** Many crosses  
**Result:** Population frequencies emerge!

### Key Insight

When you have:
- Small numbers → Think **ratios** (3:1, 1:2:1)
- Large numbers → Think **frequencies** (0.75, 0.25)

**They're the same thing, just different scales!**

---

### 🎚️ Interactive: Watch Ratios Become Frequencies


In [ ]:
@interact(
    n_crosses=IntSlider(min=1, max=1000, step=1, value=1, 
                       description='# Crosses:', style={'description_width': 'initial'}),
    parent1_geno=Dropdown(options=['AA', 'Aa', 'aa'], value='Aa', 
                         description='Parent 1:', style={'description_width': 'initial'}),
    parent2_geno=Dropdown(options=['AA', 'Aa', 'aa'], value='Aa', 
                         description='Parent 2:', style={'description_width': 'initial'}),
    show_expected=Checkbox(value=True, description='Show expected frequencies')
)
def simulate_crosses_to_population(n_crosses, parent1_geno, parent2_geno, show_expected):
    """
    Simulate multiple crosses and watch ratios become frequencies
    """
    # Define gamete production
    def get_gametes(genotype):
        if genotype == 'AA':
            return ['A', 'A']
        elif genotype == 'Aa':
            return ['A', 'a']
        else:  # aa
            return ['a', 'a']
    
    # Simulate crosses
    offspring_genotypes = []
    
    for _ in range(n_crosses):
        # Random gametes from each parent
        gamete1 = np.random.choice(get_gametes(parent1_geno))
        gamete2 = np.random.choice(get_gametes(parent2_geno))
        
        # Form offspring
        alleles = sorted([gamete1, gamete2], reverse=True)
        offspring = ''.join(alleles)
        offspring_genotypes.append(offspring)
    
    # Count genotypes
    from collections import Counter
    counts = Counter(offspring_genotypes)
    
    # Calculate observed frequencies
    obs_AA = counts.get('AA', 0) / n_crosses
    obs_Aa = counts.get('Aa', 0) / n_crosses
    obs_aa = counts.get('aa', 0) / n_crosses
    
    # Calculate expected frequencies (Mendelian ratios)
    gametes1 = get_gametes(parent1_geno)
    gametes2 = get_gametes(parent2_geno)
    
    expected = {'AA': 0, 'Aa': 0, 'aa': 0}
    for g1 in gametes1:
        for g2 in gametes2:
            alleles = sorted([g1, g2], reverse=True)
            geno = ''.join(alleles)
            expected[geno] += 0.25
    
    exp_AA = expected['AA']
    exp_Aa = expected['Aa']
    exp_aa = expected['aa']
    
    # Visualization
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    
    # 1. Bar chart
    ax1 = axes[0]
    genotypes = ['AA', 'Aa', 'aa']
    observed = [obs_AA, obs_Aa, obs_aa]
    expected_vals = [exp_AA, exp_Aa, exp_aa]
    
    x = np.arange(len(genotypes))
    width = 0.35
    
    bars1 = ax1.bar(x - width/2, observed, width, label='Observed', 
                    color='steelblue', alpha=0.8, edgecolor='black')
    
    if show_expected:
        bars2 = ax1.bar(x + width/2, expected_vals, width, label='Expected (Mendelian)', 
                       color='coral', alpha=0.8, edgecolor='black')
    
    ax1.set_ylabel('Frequency', fontsize=12, fontweight='bold')
    ax1.set_title(f'Genotype Frequencies (n={n_crosses} crosses)', fontsize=13, fontweight='bold')
    ax1.set_xticks(x)
    ax1.set_xticklabels(genotypes, fontsize=11, fontweight='bold')
    ax1.legend(fontsize=10)
    ax1.set_ylim(0, 1)
    ax1.grid(True, alpha=0.3, axis='y')
    
    # Add values on bars
    for bar, val in zip(bars1, observed):
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., height + 0.02,
                f'{val:.3f}', ha='center', fontsize=10, fontweight='bold')
    
    # 2. Show transition from ratios to frequencies
    ax2 = axes[1]
    ax2.axis('off')
    
    if n_crosses == 1:
        text = f"""ONE CROSS
        
Cross: {parent1_geno} × {parent2_geno}

Result: {offspring_genotypes[0]}

Think: Individual outcomes
"What genotype did I get?"
"""
    elif n_crosses < 10:
        text = f"""FEW CROSSES (n={n_crosses})
        
Results: {', '.join(offspring_genotypes)}

Think: Small sample ratios
"Do I see the expected ratio?"

May not match perfectly!
"""
    elif n_crosses < 100:
        text = f"""MANY CROSSES (n={n_crosses})
        
AA: {counts.get('AA', 0)}
Aa: {counts.get('Aa', 0)}
aa: {counts.get('aa', 0)}

Transition: Ratios → Frequencies
Getting closer to expected!
"""
    else:
        text = f"""POPULATION (n={n_crosses})
        
Freq(AA) = {obs_AA:.3f}
Freq(Aa) = {obs_Aa:.3f}
Freq(aa) = {obs_aa:.3f}

Think: Population frequencies
"What proportion has each genotype?"

Welcome to Population Genetics!
"""
    
    ax2.text(0.5, 0.5, text, ha='center', va='center', fontsize=11,
            bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8),
            family='monospace')
    
    # 3. Allele frequencies
    ax3 = axes[2]
    
    # Count alleles
    n_A = counts.get('AA', 0) * 2 + counts.get('Aa', 0)
    n_a = counts.get('aa', 0) * 2 + counts.get('Aa', 0)
    total_alleles = n_A + n_a
    
    p = n_A / total_alleles if total_alleles > 0 else 0
    q = n_a / total_alleles if total_alleles > 0 else 0
    
    # Pie chart
    if total_alleles > 0:
        colors = ['#FF6B6B', '#4ECDC4']
        explode = (0.05, 0.05)
        wedges, texts, autotexts = ax3.pie([p, q], labels=['A', 'a'], 
                                            autopct='%1.1f%%',
                                            colors=colors, explode=explode,
                                            startangle=90, textprops={'fontsize': 12, 'fontweight': 'bold'})
        ax3.set_title(f'Allele Frequencies\np(A)={p:.3f}, q(a)={q:.3f}', 
                     fontsize=13, fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    # Print interpretation
    print("\n📊 Analysis:")
    print("=" * 70)
    print(f"\nCross: {parent1_geno} × {parent2_geno}")
    print(f"Number of offspring: {n_crosses}")
    
    if n_crosses == 1:
        print("\n→ MENDELIAN VIEW: Looking at individual outcomes")
        print(f"   Offspring genotype: {offspring_genotypes[0]}")
    elif n_crosses < 20:
        print("\n→ TRANSITION: Starting to see patterns")
        print(f"   Observed counts: AA={counts.get('AA', 0)}, Aa={counts.get('Aa', 0)}, aa={counts.get('aa', 0)}")
        print(f"   Expected ratio: AA:{exp_AA*4:.1f}, Aa:{exp_Aa*4:.1f}, aa:{exp_aa*4:.1f}")
    else:
        print("\n→ POPULATION VIEW: Thinking in frequencies")
        print(f"   Genotype frequencies:")
        print(f"      f(AA) = {obs_AA:.3f} (expected: {exp_AA:.3f})")
        print(f"      f(Aa) = {obs_Aa:.3f} (expected: {exp_Aa:.3f})")
        print(f"      f(aa) = {obs_aa:.3f} (expected: {exp_aa:.3f})")
        print(f"\n   Allele frequencies:")
        print(f"      p(A) = {p:.3f}")
        print(f"      q(a) = {q:.3f}")
        print(f"      p + q = {p+q:.3f} ✓")
    
    print("\n💡 Key Insight:")
    if n_crosses == 1:
        print("   With one cross, we think about THIS offspring.")
        print("   Try increasing n_crosses to see the transition!")
    elif n_crosses < 50:
        print("   With few crosses, we see RATIOS emerging from randomness.")
        print("   Notice: Observed may not match expected perfectly (sampling variation!)")
    else:
        print("   With many crosses, we think about FREQUENCIES in the population.")
        print("   This is Population Genetics! Same rules, different scale.")
    
    print("\n" + "=" * 70)

### 💡 Try This:

**Experiment 1: Watch the Transition**
- Set cross to Aa × Aa
- Start with n=1: See individual outcome
- Try n=4: See if you get 3:1 ratio
- Try n=10: Notice variation
- Try n=100: See frequencies stabilize
- Try n=1000: Perfect match with expectations!

**Experiment 2: Different Crosses**
- AA × aa: Always get Aa (no variation!)
- Aa × aa: 1:1 ratio becomes 0.5:0.5 frequencies
- aa × aa: All aa (p=0, q=1)

**Key Learning:** Same genetics, different way of thinking!

---

## 🎯 Part 2: Hardy-Weinberg Equilibrium - What It Really Means

### The Famous Equation

$$p^2 + 2pq + q^2 = 1$$

### What Students Think It Means:
- "Some complicated formula I have to memorize"
- "Something about equilibrium"
- "No idea how it relates to Punnett squares"

### What It Actually Means:

**It's just a BIG Punnett square for an entire population!**

```
Individual Cross:       Population Mating:
     A    a                    A(p)   a(q)
   +----+----+              +------+------+
A  | AA | Aa |           A(p)| p²  | pq  |
   +----+----+              +------+------+
a  | Aa | aa |           a(q)| pq  | q²  |
   +----+----+              +------+------+

Ratio: 1:2:1            Frequency: p²:2pq:q²
```

**Same logic! Just scaled up!**

### The Five Conditions

Hardy-Weinberg holds when:
1. **Large population** (no genetic drift)
2. **Random mating** (no inbreeding/assortative mating)
3. **No mutations** (alleles don't change)
4. **No migration** (no gene flow)
5. **No selection** (all genotypes equally fit)

**When ALL five are true → frequencies don't change!**

---

### 🎚️ Interactive: Hardy-Weinberg Explorer


In [ ]:
@interact(
    p_freq=FloatSlider(min=0.0, max=1.0, step=0.01, value=0.6, 
                      description='p (freq of A):', style={'description_width': 'initial'}),
    pop_size=IntSlider(min=50, max=10000, step=50, value=1000,
                      description='Population size:', style={'description_width': 'initial'}),
    show_punnett=Checkbox(value=True, description='Show population Punnett square')
)
def hardy_weinberg_explorer(p_freq, pop_size, show_punnett):
    """
    Explore Hardy-Weinberg equilibrium predictions
    """
    # Calculate q
    q_freq = 1 - p_freq
    
    # HW predictions
    hw_AA = p_freq ** 2
    hw_Aa = 2 * p_freq * q_freq
    hw_aa = q_freq ** 2
    
    # Simulate random mating population
    # Each individual gets two alleles drawn from population allele pool
    alleles = np.random.choice(['A', 'a'], size=(pop_size, 2), 
                              p=[p_freq, q_freq])
    
    # Count genotypes
    genotypes = []
    for individual in alleles:
        geno = ''.join(sorted(individual, reverse=True))
        genotypes.append(geno)
    
    from collections import Counter
    counts = Counter(genotypes)
    
    obs_AA = counts.get('AA', 0) / pop_size
    obs_Aa = counts.get('Aa', 0) / pop_size
    obs_aa = counts.get('aa', 0) / pop_size
    
    # Visualization
    if show_punnett:
        fig = plt.figure(figsize=(16, 10))
        gs = fig.add_gridspec(3, 2, hspace=0.3, wspace=0.3)
    else:
        fig = plt.figure(figsize=(16, 8))
        gs = fig.add_gridspec(2, 2, hspace=0.3, wspace=0.3)
    
    # 1. Population Punnett Square
    if show_punnett:
        ax1 = fig.add_subplot(gs[0, :])
        ax1.set_xlim(0, 1)
        ax1.set_ylim(0, 1)
        ax1.axis('off')
        
        # Draw grid
        cell_width = 0.3
        start_x = 0.2
        start_y = 0.3
        
        # Labels
        ax1.text(start_x + 0.5*cell_width, start_y + 2.2*cell_width, 
                f'Female Gametes', ha='center', fontsize=13, fontweight='bold')
        ax1.text(start_x - 0.15, start_y + cell_width, 
                'Male\nGametes', ha='center', va='center', fontsize=13, fontweight='bold')
        
        # Column headers
        ax1.text(start_x + 0.5*cell_width, start_y + 1.7*cell_width,
                f'A ({p_freq:.2f})', ha='center', fontsize=12, fontweight='bold')
        ax1.text(start_x + 1.5*cell_width, start_y + 1.7*cell_width,
                f'a ({q_freq:.2f})', ha='center', fontsize=12, fontweight='bold')
        
        # Row headers
        ax1.text(start_x - 0.1, start_y + 1.5*cell_width,
                f'A ({p_freq:.2f})', ha='right', fontsize=12, fontweight='bold')
        ax1.text(start_x - 0.1, start_y + 0.5*cell_width,
                f'a ({q_freq:.2f})', ha='right', fontsize=12, fontweight='bold')
        
        # Cells with genotypes and frequencies
        cells = [
            (start_x, start_y + cell_width, 'AA', hw_AA, '#FFB6C6'),
            (start_x + cell_width, start_y + cell_width, 'Aa', p_freq*q_freq, '#B6D7FF'),
            (start_x, start_y, 'Aa', p_freq*q_freq, '#B6D7FF'),
            (start_x + cell_width, start_y, 'aa', hw_aa, '#C1FFC1')
        ]
        
        for x, y, geno, freq, color in cells:
            rect = Rectangle((x, y), cell_width, cell_width, 
                           facecolor=color, edgecolor='black', linewidth=2)
            ax1.add_patch(rect)
            ax1.text(x + cell_width/2, y + cell_width/2 + 0.05, geno,
                    ha='center', fontsize=16, fontweight='bold')
            ax1.text(x + cell_width/2, y + cell_width/2 - 0.05, f'{freq:.3f}',
                    ha='center', fontsize=12)
        
        # Summary
        summary_text = f"""Hardy-Weinberg Prediction:
        
AA = p² = ({p_freq:.2f})² = {hw_AA:.3f}
Aa = 2pq = 2({p_freq:.2f})({q_freq:.2f}) = {hw_Aa:.3f}
aa = q² = ({q_freq:.2f})² = {hw_aa:.3f}

Sum = {hw_AA + hw_Aa + hw_aa:.3f} ✓"""
        
        ax1.text(0.75, 0.5, summary_text, fontsize=11, family='monospace',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
        
        ax1.set_title('Population Punnett Square (Random Mating)', 
                     fontsize=14, fontweight='bold', pad=20)
        
        row_offset = 1
    else:
        row_offset = 0
    
    # 2. Observed vs Expected
    ax2 = fig.add_subplot(gs[row_offset, 0])
    
    genos = ['AA', 'Aa', 'aa']
    observed = [obs_AA, obs_Aa, obs_aa]
    expected = [hw_AA, hw_Aa, hw_aa]
    
    x = np.arange(len(genos))
    width = 0.35
    
    bars1 = ax2.bar(x - width/2, observed, width, label='Observed (simulated)', 
                    color='steelblue', alpha=0.8, edgecolor='black')
    bars2 = ax2.bar(x + width/2, expected, width, label='HW Expected', 
                   color='coral', alpha=0.8, edgecolor='black')
    
    ax2.set_ylabel('Frequency', fontsize=12, fontweight='bold')
    ax2.set_title(f'Genotype Frequencies (N={pop_size})', fontsize=13, fontweight='bold')
    ax2.set_xticks(x)
    ax2.set_xticklabels(genos, fontsize=11, fontweight='bold')
    ax2.legend(fontsize=10)
    ax2.set_ylim(0, max(max(observed), max(expected)) * 1.2)
    ax2.grid(True, alpha=0.3, axis='y')
    
    # 3. Chi-square test
    ax3 = fig.add_subplot(gs[row_offset, 1])
    ax3.axis('off')
    
    # Calculate chi-square
    obs_counts = [counts.get('AA', 0), counts.get('Aa', 0), counts.get('aa', 0)]
    exp_counts = [hw_AA * pop_size, hw_Aa * pop_size, hw_aa * pop_size]
    
    chi_sq = sum((o - e)**2 / e for o, e in zip(obs_counts, exp_counts) if e > 0)
    df = 1  # 3 genotypes - 1 allele frequency estimated - 1
    p_value = 1 - chi2.cdf(chi_sq, df)
    
    test_text = f"""Chi-Square Goodness of Fit Test:
    
H₀: Population in HW equilibrium

χ² = {chi_sq:.3f}
df = {df}
p-value = {p_value:.4f}

Interpretation:
"""
    
    if p_value > 0.05:
        test_text += "✓ Cannot reject H₀\n"
        test_text += "Population appears to be\nin Hardy-Weinberg equilibrium\n"
        test_text += "(p > 0.05)"
        color = 'lightgreen'
    else:
        test_text += "✗ Reject H₀\n"
        test_text += "Population deviates from\nHW equilibrium\n"
        test_text += "(p < 0.05)"
        color = 'lightcoral'
    
    ax3.text(0.5, 0.5, test_text, ha='center', va='center', fontsize=11,
            bbox=dict(boxstyle='round', facecolor=color, alpha=0.8),
            family='monospace')
    
    ax3.set_title('Statistical Test', fontsize=13, fontweight='bold')
    
    # 4. Allele frequencies
    ax4 = fig.add_subplot(gs[row_offset + 1, :])
    
    # Calculate observed allele frequencies from genotypes
    obs_p = (2*counts.get('AA', 0) + counts.get('Aa', 0)) / (2*pop_size)
    obs_q = (2*counts.get('aa', 0) + counts.get('Aa', 0)) / (2*pop_size)
    
    alleles_data = [
        ['', 'Expected', 'Observed'],
        ['p (A)', f'{p_freq:.4f}', f'{obs_p:.4f}'],
        ['q (a)', f'{q_freq:.4f}', f'{obs_q:.4f}'],
        ['Sum', f'{p_freq + q_freq:.4f}', f'{obs_p + obs_q:.4f}']
    ]
    
    table = ax4.table(cellText=alleles_data, cellLoc='center',
                     bbox=[0.3, 0.2, 0.4, 0.6])
    table.auto_set_font_size(False)
    table.set_fontsize(12)
    table.scale(1, 2)
    
    for i in range(len(alleles_data)):
        for j in range(len(alleles_data[i])):
            if i == 0 or j == 0:
                table[(i, j)].set_facecolor('#E8E8E8')
                table[(i, j)].set_text_props(weight='bold')
    
    ax4.axis('off')
    ax4.set_title('Allele Frequencies Check', fontsize=13, fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    # Print summary
    print("\n🧬 Hardy-Weinberg Analysis:")
    print("=" * 70)
    print(f"\nPopulation size: {pop_size}")
    print(f"Allele frequency p(A) = {p_freq:.3f}")
    print(f"Allele frequency q(a) = {q_freq:.3f}")
    
    print(f"\nHardy-Weinberg Predictions:")
    print(f"  AA = p² = {hw_AA:.4f} (expected {int(hw_AA*pop_size)} individuals)")
    print(f"  Aa = 2pq = {hw_Aa:.4f} (expected {int(hw_Aa*pop_size)} individuals)")
    print(f"  aa = q² = {hw_aa:.4f} (expected {int(hw_aa*pop_size)} individuals)")
    
    print(f"\nObserved Frequencies (from simulation):")
    print(f"  AA = {obs_AA:.4f} (observed {counts.get('AA', 0)} individuals)")
    print(f"  Aa = {obs_Aa:.4f} (observed {counts.get('Aa', 0)} individuals)")
    print(f"  aa = {obs_aa:.4f} (observed {counts.get('aa', 0)} individuals)")
    
    print(f"\n💡 Key Insight:")
    if abs(obs_AA - hw_AA) < 0.02 and abs(obs_Aa - hw_Aa) < 0.02:
        print("   Excellent match! Random mating in large population produces")
        print("   Hardy-Weinberg genotype frequencies.")
    else:
        print("   Small deviation due to sampling variation (random chance).")
        print("   Try increasing population size for better match!")
    
    print("\n" + "=" * 70)

### 🧪 Experiments:

**1. Effect of Population Size**
- Set p=0.6, N=50: Notice variation
- Set p=0.6, N=1000: Much better match
- Set p=0.6, N=10000: Nearly perfect!

**2. Different Allele Frequencies**
- p=0.5: Most heterozygotes (2pq maximum)
- p=0.9: Mostly AA (dominant allele common)
- p=0.1: Mostly aa (recessive allele common)

**3. Extreme Cases**
- p=1.0: Only AA exists (allele a lost)
- p=0.0: Only aa exists (allele A lost)

---

## 🎯 Summary & Key Takeaways

### What You've Learned:

1. ✅ **Mendelian → Population Genetics**
   - Same principles, different scale
   - Ratios (small n) → Frequencies (large n)
   - Individual focus → Population focus

2. ✅ **Hardy-Weinberg is Just a Big Punnett Square**
   - p² + 2pq + q² = 1
   - Predicts genotype frequencies from allele frequencies
   - Baseline for detecting evolution

3. ✅ **Why Study Populations Statistically?**
   - Evolution happens to populations, not individuals
   - Frequencies change, not genotypes
   - Understanding variation requires statistics

4. ✅ **The Five Conditions Matter**
   - Large size, random mating, no mutation, no migration, no selection
   - When violated → evolution occurs!
   - HW is a null hypothesis

### The Bridge:

**Mendelian genetics tells us what happens in ONE cross.**  
**Population genetics tells us what happens in MANY crosses over TIME.**

Both use the same fundamental rules!  
The only difference is scale and perspective.

---

**Authors:** Susama Kar & Dr. Alok Patel  
**Institution:** Kuchinda College, Sambalpur University  
**License:** CC BY 4.0  
**Repository:** github.com/The-Pattern-Hunter/principles-of-genetics-interactive
